# AULA 22/10 — REVISÃO ORIENTADA PARA O MINI-PROJETO M1.1
## Sprints 2, 3 e 4: limpeza, funções e análise

Hoje vamos praticar os pontos que mais costumam gerar dúvida:
- padronização de texto;
- tratamento de ausências;
- remoção de duplicatas confirmadas;
- criação de **duas funções com parâmetro e retorno**;
- filtro;
- estatísticas básicas;
- dois `groupby()` com interpretação.

# Importação e leitura do Dataframe

In [16]:
from google.colab import files
uploaded = files.upload()

import pandas as pd

arquivo = next(iter(uploaded))

df_original = pd.read_csv(arquivo)

df = df_original.copy()

Saving clientes_limpeza_funcoes.csv to clientes_limpeza_funcoes (1).csv


## Estrutura do DataFrame



In [17]:
display(df_original.head())

print("Dimensões: ", df_original.shape)
print("Colunas: ", df_original.columns.tolist())

df_original.info()

,cliente_id,genero,cidade,categoria_preferida,valor_compra,quantidade_itens,satisfacao
0,C001,Feminino,Belo Horizonte,Casa,527.26,7,2.0
1,C002,feminino,Belo Horizonte,Eletrônicos,182.24,1,5.0
2,C003,Feminino,Curitiba,Casa,357.35,4,3.0
3,C004,Masculino,Curitiba,Moda,692.27,6,3.0
4,C005,Masculino,rio de janeiro,Casa,628.30,4,3.0


Dimensões:  (35, 7)
Colunas:  ['cliente_id', 'genero', 'cidade', 'categoria_preferida', 'valor_compra', 'quantidade_itens', 'satisfacao']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   cliente_id           35 non-null     object 
 1   genero               35 non-null     object 
 2   cidade               35 non-null     object 
 3   categoria_preferida  35 non-null     object 
 4   valor_compra         33 non-null     float64
 5   quantidade_itens     35 non-null     int64  
 6   satisfacao           34 non-null     float64
dtypes: float64(2), int64(1), object(4)
memory usage: 2.0+ KB


## ETAPA 1 — Conferindo problemas antes de escolher a solução

In [18]:
print("Ausência por coluna: ")
display(df_original.isna().sum())

print(
    "Duplicatas completas excedentes: ",
    df_original.duplicated().sum()
)

Ausência por coluna: 


,0
cliente_id,0
genero,0
cidade,0
categoria_preferida,0
valor_compra,2
quantidade_itens,0
satisfacao,1


Duplicatas completas excedentes:  1


In [19]:
# verificar os espaços

colunas_texto = [
    "cliente_id",
    "genero",
    "cidade",
    "categoria_preferida"
]

for coluna in colunas_texto:
  quantidade_em_branco = (
      df[coluna]
      .astype("string")
      .str.strip()
      .eq("")
      .sum()
  )

  print(coluna, ":", quantidade_em_branco)

cliente_id : 0
genero : 0
cidade : 0
categoria_preferida : 0


In [20]:
# Visualizando diferenças de escrita

for coluna in ["genero", "cidade", "categoria_preferida"]:
  print(coluna)
  print(df_original[coluna].unique())

genero
['Feminino' ' feminino ' 'Masculino' 'Não informado' 'FEMININO']
cidade
['Belo Horizonte' 'Curitiba' 'rio de janeiro' 'São Paulo' 'Rio de Janeiro'
 'SÃO PAULO ']
categoria_preferida
['Casa' 'Eletrônicos' 'Moda' 'eletronicos' 'Beleza']


In [21]:
df_original.head(16)

,cliente_id,genero,cidade,categoria_preferida,valor_compra,quantidade_itens,satisfacao
0,C001,Feminino,Belo Horizonte,Casa,527.26,7,2.0
1,C002,feminino,Belo Horizonte,Eletrônicos,182.24,1,5.0
2,C003,Feminino,Curitiba,Casa,357.35,4,3.0
3,C004,Masculino,Curitiba,Moda,692.27,6,3.0
4,C005,Masculino,rio de janeiro,Casa,628.30,4,3.0
5,C006,Não informado,São Paulo,Casa,575.10,1,4.0
6,C007,Masculino,Rio de Janeiro,eletronicos,631.81,6,2.0
7,C008,Masculino,São Paulo,Moda,144.23,1,3.0
8,C009,FEMININO,Belo Horizonte,Moda,559.73,2,3.0
9,C010,Masculino,Curitiba,Casa,875.44,2,4.0


# Sprint 2: Decisões

| Problema | Decisão adotada | Justificativa |
|---|---|---|
| Maiúsculas, minúsculas e espaços | Padronizar os textos | Evitar separar o mesmo grupo em categorias diferentes |
| `eletronicos` sem acento | Substituir por `Eletrônicos` | A grafia representa a mesma categoria |
| Dois valores de compra ausentes | Manter como ausentes | Não há informação suficiente para recuperar o valor |
| Uma satisfação ausente | Manter como ausente | As demais informações do registro continuam úteis |
| Uma cópia completa do registro `C006` | Remover a cópia excedente | Evitar contar duas vezes o mesmo registro |


## Função de tratamento - Sprint 3

No projeto, uma função precisa:
1. receber um dado por parâmetro;
2. executar uma tarefa útil;
3. devolver um resultado com `return`.

Crie uma função para padronizar texto.

In [22]:
def padronizar_texto(valor):
  if pd.isna(valor):
    return pd.NA

  texto = str(valor).strip().lower()

  if texto == "":
    return pd.NA

  ajustes = {
      "eletronicos": "Eletrônicos",
      "rio de janeiro": "Rio de Janeiro",
      "não informado": "Não informado"
  }

  return ajustes.get(texto, texto.title())

In [8]:
print(padronizar_texto(" feminino "))
print(padronizar_texto("eletronicos"))
print(padronizar_texto("rio de janeiro"))
print(padronizar_texto("  "))

Feminino
Eletrônicos
Rio de Janeiro
<NA>


In [23]:
for coluna in ["genero", "cidade", "categoria_preferida"]:
  df[coluna] = df[coluna].apply(padronizar_texto)

In [24]:
for coluna in ["genero", "cidade", "categoria_preferida"]:
  print(coluna)
  print(df[coluna].unique())

genero
['Feminino' 'Masculino' 'Não informado']
cidade
['Belo Horizonte' 'Curitiba' 'Rio de Janeiro' 'São Paulo']
categoria_preferida
['Casa' 'Eletrônicos' 'Moda' 'Beleza']


# Aplicar a decisão sobre duplicatas e comparar os resultados

In [25]:
display(
  df[df.duplicated(keep=False)]
)

,cliente_id,genero,cidade,categoria_preferida,valor_compra,quantidade_itens,satisfacao
5,C006,Não informado,São Paulo,Casa,575.1,1,4.0
34,C006,Não informado,São Paulo,Casa,575.1,1,4.0


In [26]:
df = df.drop_duplicates().copy()

In [28]:
print("Linhas antes: ", len(df_original))
print("Linhas depois: ", len(df))

print(
    "Duplicatas restantes: ",
    df.duplicated().sum()
)
display(df.isna().sum())

Linhas antes:  35
Linhas depois:  34
Duplicatas restantes:  0


,0
cliente_id,0
genero,0
cidade,0
categoria_preferida,0
valor_compra,2
quantidade_itens,0
satisfacao,1


### A base passou de 35 para 34 registros após a remoção de uma repetição completa. Foram padronizadas 3 colunas textuais. As ausências de valor de compra e satisfação foram preservadas.

## Segunda função: resumo numérico

Crie uma função que receba uma Series numérica e retorne um dicionário com:
- contagem válida;
- média;
- mínimo;
- máximo.

In [29]:
def resumo_numerico(serie):
  return {
      "contagem_valida": serie.count(),
      "media": serie.mean(),
      "minimo": serie.min(),
      "maximo": serie.max()
  }


In [31]:
resumo = resumo_numerico(df["valor_compra"])

print("Valores válidos: ", resumo["contagem_valida"])
print("Média: ", round(resumo["media"], 2))
# Acessando com as chaves acentuadas conforme gerado pela função
print("Mínimo: ", resumo.get("minimo", resumo.get("mínimo")))
print("Máximo: ", resumo.get("maximo", resumo.get("máximo")))

Valores válidos:  32
Média:  649.78
Mínimo:  144.23
Máximo:  1188.05


### Dos 34 registros tratados, 32 possuem valor de compra preenchido. Entre esses 32 valores conhecidos, a média do valor de compra registrado é de aproximadamente R$ 649,78.

## Sprint 4 — Transformar uma pergunta em filtro

Escolha um filtro com significado de negócio.

Exemplo de pergunta:
> “Quais registros representam clientes com compras acima de R$ 500?”

In [32]:
compras_acima_500 = df[df["valor_compra"] > 500]

display(compras_acima_500)

print(
    "Quantidade de registros: ",
    len(compras_acima_500)
)

,cliente_id,genero,cidade,categoria_preferida,valor_compra,quantidade_itens,satisfacao
0,C001,Feminino,Belo Horizonte,Casa,527.26,7,2.0
3,C004,Masculino,Curitiba,Moda,692.27,6,3.0
4,C005,Masculino,Rio de Janeiro,Casa,628.30,4,3.0
5,C006,Não informado,São Paulo,Casa,575.10,1,4.0
6,C007,Masculino,Rio de Janeiro,Eletrônicos,631.81,6,2.0
8,C009,Feminino,Belo Horizonte,Moda,559.73,2,3.0
9,C010,Masculino,Curitiba,Casa,875.44,2,4.0
11,C012,Não informado,Curitiba,Casa,852.38,5,5.0
12,C013,Feminino,São Paulo,Moda,913.87,7,4.0
13,C014,Feminino,Rio de Janeiro,Moda,726.40,7,1.0


Quantidade de registros:  23


In [33]:
# Percentual de registros acima de R$ 500

percentual = (
    len(compras_acima_500)
    / df["valor_compra"].count() * 100
)

print(round(percentual, 2))

71.88


### Usamos 32 porque estamos calculando o percentual entre os registros cujo valor conhecemos. Os dois valores ausentes não permitem decidir se a compra ultrapassa R$ 500.

## ETAPA 7 — Dois agrupamentos

Faça dois resumos diferentes.

Exemplos:
- quantidade de registros por `categoria_preferida`;
- soma de `valor_compra` por `genero`.

Depois escreva, em português, **o que cada resultado mede**.

## Primeiro agrupamento: registros por categoria preferida

Como os registros estão distribuídos entre as categorias preferidas?

In [34]:
registros_por_categoria = (
    df.groupby("categoria_preferida")
    .size()
    .sort_values(ascending=False)
)
display(registros_por_categoria)

,0
categoria_preferida,
Casa,10
Moda,9
Beleza,8
Eletrônicos,7


### Casa é a categoria preferida mais frequente nesta base, aparecendo em 10 de 34 registros

# Segundo agrupamento: valores por gênero

Qual é a soma dos valores de compra conhecidos em cada grupo de gênero registrado?

In [36]:
grupos_genero = (
    df.groupby("genero", dropna=False)["valor_compra"]
)

resumo_por_genero = pd.DataFrame({
    "registros": grupos_genero.size(),
    "valores_validos": grupos_genero.count(),
    "soma_valores_conhecidos": grupos_genero.sum(min_count=1)

})

display(resumo_por_genero.round(2))

,registros,valores_validos,soma_valores_conhecidos
genero,,,
Feminino,11,11,6477.43
Masculino,10,9,5371.44
Não informado,13,12,8944.03


## Interpretação final

1. Uma estatística com sua unidade.

A média dos 32 valores de compra conhecidosé de R$ 649,78

2. Uma interpretação com filtro.

Existem 23 registros com valor de compra superior a R$ 500.

3. Uma interpretação de primeiro agrupamento

Casa aparece como categoria preferida em 10 registros.

4. Uma interpretação do segundo agrupamento, com mais informações.

Os valores de compra conhecidos nos registros classificados como Feminino somam R$ 6.477,43.

